Thuật toán svm hay randomforest không biết chữ nên cần biến đồi thành số vector hóa

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC  # NHANH HƠN 10-50 lần so với SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score
import joblib
import os
import time

print("🚀 === SO SÁNH THUẬT TOÁN NHANH === 🚀\n")

print("1. Đang đọc dữ liệu đã làm giàu...")
df_enriched = pd.read_csv("../data/processed/df_enriched.csv")

# Xử lý các ô trống (NaN) trong trường hợp WordNet/DBpedia không tìm thấy đặc trưng nào
df_enriched = df_enriched.fillna("")

print("2. Đang gộp văn bản và vector hóa (TF-IDF)...")
# Gộp 3 cột lại thành một chuỗi văn bản dài mang đầy đủ ngữ nghĩa
df_enriched['combined_text'] = (df_enriched['text'] + " " + 
                                df_enriched['wordnet_features'] + " " + 
                                df_enriched['dbpedia_features'])

# Chuyển đổi văn bản thành ma trận số bằng TF-IDF (GIẢM xuống 3000 để nhanh hơn)
vectorizer = TfidfVectorizer(max_features=3000) 
X = vectorizer.fit_transform(df_enriched['combined_text'])
y = df_enriched['label']

print("3. Đang chia tập dữ liệu Train/Test...")
# Chia 80% để học, 20% để làm bài kiểm tra đánh giá
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Thử 3 thuật toán NHANH NHẤT
models_to_test = {
    '⚡ Naive Bayes (Cực Nhanh)': MultinomialNB(),
    '🚀 Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    '💨 LinearSVC (SVM Nhanh)': LinearSVC(max_iter=2000, random_state=42)
}

results = []
best_model = None
best_accuracy = 0

for name, model in models_to_test.items():
    print(f"\n4. Đang huấn luyện: {name}...")
    
    # Đo thời gian huấn luyện
    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time
    
    # Đánh giá
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    
    print(f"   ⏱️  Thời gian: {training_time:.2f}s ({training_time/60:.2f} phút)")
    print(f"   🎯 Accuracy: {accuracy:.4f}")
    
    results.append({
        'Thuật toán': name,
        'Accuracy': accuracy,
        'Thời gian (giây)': training_time
    })
    
    # Lưu model tốt nhất
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model = model
        best_model_name = name

print("\n" + "="*70)
print("📊 BẢNG KẾT QUẢ SO SÁNH")
print("="*70)
results_df = pd.DataFrame(results)
for idx, row in results_df.iterrows():
    print(f"{row['Thuật toán']:35} | Accuracy: {row['Accuracy']:.4f} | {row['Thời gian (giây)']:.2f}s")

print("\n" + "="*70)
print(f"🏆 Model tốt nhất: {best_model_name}")
print(f"🎯 Accuracy: {best_accuracy:.4f}")
print("="*70)

print("\n--- CHI TIẾT ĐÁNH GIÁ MODEL TỐT NHẤT ---")
best_predictions = best_model.predict(X_test)
print(classification_report(y_test, best_predictions))

# 6. LƯU MÔ HÌNH (Export)
# Tạo thư mục models nếu chưa có
os.makedirs("../models", exist_ok=True)

# Lưu cả mô hình và bộ chuyển đổi vectorizer để dùng cho Giao diện sau này
joblib.dump(best_model, "../models/best_classifier.pkl")
joblib.dump(vectorizer, "../models/tfidf_vectorizer.pkl")

print(f"\n💾 Đã lưu thành công model '{best_model_name}' vào thư mục models/!")

🚀 === SO SÁNH THUẬT TOÁN NHANH === 🚀

1. Đang đọc dữ liệu đã làm giàu...
2. Đang gộp văn bản và vector hóa (TF-IDF)...
3. Đang chia tập dữ liệu Train/Test...

4. Đang huấn luyện: ⚡ Naive Bayes (Cực Nhanh)...
   ⏱️  Thời gian: 0.03s (0.00 phút)
   🎯 Accuracy: 0.8717

4. Đang huấn luyện: 🚀 Logistic Regression...
   ⏱️  Thời gian: 5.14s (0.09 phút)
   🎯 Accuracy: 0.8960

4. Đang huấn luyện: 💨 LinearSVC (SVM Nhanh)...
   ⏱️  Thời gian: 6.97s (0.12 phút)
   🎯 Accuracy: 0.8980

📊 BẢNG KẾT QUẢ SO SÁNH
⚡ Naive Bayes (Cực Nhanh)           | Accuracy: 0.8717 | 0.03s
🚀 Logistic Regression               | Accuracy: 0.8960 | 5.14s
💨 LinearSVC (SVM Nhanh)             | Accuracy: 0.8980 | 6.97s

🏆 Model tốt nhất: 💨 LinearSVC (SVM Nhanh)
🎯 Accuracy: 0.8980

--- CHI TIẾT ĐÁNH GIÁ MODEL TỐT NHẤT ---
              precision    recall  f1-score   support

           1       0.92      0.88      0.90      5956
           2       0.94      0.97      0.96      6058
           3       0.86      0.86      0.86 